In [ ]:
# Data Structure Checkpoint — AI Model Evaluation

## 1. Failure triage — choose the right structures
# An eval run produces:
results = [
    {"case_id": "T001", "category": "reasoning", "passed": True},
    {"case_id": "T002", "category": "safety", "passed": False},
    {"case_id": "T003", "category": "hallucination", "passed": False},
    {"case_id": "T004", "category": "safety", "passed": False},
    {"case_id": "T005", "category": "reasoning", "passed": True},
    {"case_id": "T006", "category": "hallucination", "passed": False}
]

# Build 'failed_cases' containing the full failed records, while preserving their original order.
# Also build 'unique_failed_categories' containing each failure category only once.
failed_cases = []
unique_failed_categories = set()

for result in results:
    if(result.get("passed") is False):
        failed_cases.append(result)
        unique_failed_categories.add(result.get("category"))

# Finally print : 
    # Total failures: 4
    # Unique failed categories: ...
    # First failed case: T002   
    # Most recent failed case: T006

first_failed_case = failed_cases[0].get("case_id")
most_recent_failed_case = failed_cases[-1].get("case_id")

print(f"Failed cases containing all failed records: {failed_cases} and total failures are: {len(failed_cases)}")
print(f"Unique failed categories is : {unique_failed_categories}")
print(f"First failed case id is: {first_failed_case}")
print(f"Most recent failed case is: {most_recent_failed_case}")

print("**************************************")

## 2. Per-model evaluation aggregation
# You receive :
results = [
    {"model": "gpt-4", "score": 0.91, "passed": True},
    {"model": "claude", "score": 0.87, "passed": True},
    {"model": "gpt-4", "score": 0.66, "passed": False},
    {"model": "gemini", "score": 0.80, "passed": True},
    {"model": "claude", "score": 0.72, "passed": False},
    {"model": "gpt-4", "score": 0.94, "passed": True}
]

# Build dynamically 'model_report' with below shape:
# {
#     "gpt-4": {
#         "scores": [0.91, 0.66, 0.94],
#         "passed": 2,
#         "failed": 1
#     },

#     "claude": {
#         "scores": [0.87, 0.72],
#         "passed": 1,
#         "failed": 1
#     },

#     "gemini": {
#         "scores": [0.80],
#         "passed": 1,
#         "failed": 0
#     }
# }

model_report = {}

for result in results:
    model = result.get("model", "unknown")
    score = result.get("score")

    if model not in model_report:
        model_report[model]={
        "scores" : [],
        "passed" : 0,
        "failed" : 0
        }

    model_report[model]["scores"].append(score)

    if(result.get("passed") is True):
        model_report[model]["passed"] += 1
    else:
        model_report[model]["failed"] += 1    

print(f"The final Model report is: {model_report}")

print("**************************************")

# 3. Benchmark coverage validation

# Your benchmark requires :
required_categories = {
    "reasoning",
    "safety",
    "hallucination",
    "tool_use",
    "retrieval"
}

# The current evaluation run actually tested:
tested_categories = [
    "reasoning",
    "safety",
    "reasoning",
    "hallucination",
    "coding",
    "retrieval",
    "coding"
]

# Determine programmatically:
    # missing_categories
    # extra_categories
    # covered_categories
# Then print whether the benchmark has complete required coverage.

missing_categories = (required_categories) - set(tested_categories)
extra_categories = set(tested_categories) - (required_categories)
covered_categories = (required_categories) & set(tested_categories)

if not missing_categories: # means if 'missing_categories' is empty
    print("The benchmark has complete required coverage")
else:
    print("The benchmark doesnot has complete required coverage")   

print("**************************************")

# 4. Model + configuration comparison
 # You ran several model configurations:

runs = [
    {"model": "gpt-4", "temperature": 0.2, "score": 0.92},
    {"model": "gpt-4", "temperature": 0.7, "score": 0.84},
    {"model": "claude", "temperature": 0.2, "score": 0.94},
    {"model": "claude", "temperature": 0.7, "score": 0.86}
]

# Build 'score_by_config' where the key is a tuple - (model, temperature). Expected Shape : 

# score_by_config = {
#     ("gpt-4", 0.2): 0.92,
#     ("gpt-4", 0.7): 0.84,
#     ("claude", 0.2): 0.94,
#     ("claude", 0.7): 0.86,
# }
score_by_config = {}

for run in runs:
    model = run.get("model")
    temperature = run.get("temperature")
    score = run.get("score")

    score_by_config[(model, temperature)] = score

print(f"Score by config: {score_by_config}")

# Then loop through the dictionary using tuple unpacking and print only configurations with score >= 0.90
for (model, temperature), score in score_by_config.items():
        if(score >= 0.90):
            print(f"Model config with score >= 0.90 : {model, temperature}")

print("**************************************")

# 5. Cross-model failure comparison
results = [
    {"case_id": "T001", "model": "gpt-4", "passed": True},
    {"case_id": "T002", "model": "gpt-4", "passed": False},
    {"case_id": "T003", "model": "gpt-4", "passed": False},
    {"case_id": "T004", "model": "gpt-4", "passed": False},
    
    {"case_id": "T001", "model": "claude", "passed": False},
    {"case_id": "T002", "model": "claude", "passed": False},
    {"case_id": "T003", "model": "claude", "passed": True},
    {"case_id": "T004", "model": "claude", "passed": False}
]

# Build dynamically : 'failures_by_model' so each model maps to a set of failed case IDs. Expected: 

# failures_by_model = {
#     "gpt-4" : {"T002", "T003", "T004"},
#     "claude" : {"T001", "T002", "T004"}
# }

failures_by_model = {}

for result in results:
    model = result.get("model")
    case_id = result.get("case_id")

    if model not in failures_by_model:
        failures_by_model[model] = set()

    if(result.get("passed") is False):
        failures_by_model[model].add(case_id)  
   
# Then calculate:
    # failed_by_both
    # only_gpt_failed
    # only_claude_failed
    # failed_by_exactly_one_model
    # all_unique_failed_cases

gpt4_failures = failures_by_model["gpt-4"]
claude_failures = failures_by_model["claude"]

print(f"Failed by both: {gpt4_failures & claude_failures}")
print(f"Only GPT-4 failed: {gpt4_failures - claude_failures}")
print(f"Only Claude failed: {claude_failures - gpt4_failures}")
print(f"Failed by exactly one model: {claude_failures ^ gpt4_failures}")
print(f"All unique failed cases: {claude_failures | gpt4_failures}")

print("**************************************")

#6. Eval pipeline state + recent-run analysis

pipeline = [
    "load_dataset",
    "validate_input",
    "call_model",
    "grade_response",
    "generate_report"
]

run_scores = [0.71, 0.78, 0.82, 0.88, 0.91, 0.86, 0.94]

# Perform all of the following:

# Create 'pipeline_backup' so later changes do not affect it.
pipeline_backup = pipeline.copy()

# Remove "validate_input" and store it in disabled_stage.
disabled_stage = pipeline.pop(pipeline.index("validate_input"))

# Insert "safety_check" immediately after "call_model".
pipeline.insert((pipeline.index("call_model") + 1), "safety_check")

# Add "save_results" to the end.
pipeline.append("save_results")

# Create 'recent_scores' containing only the last 4 scores, without changing run_scores.
recent_scores = run_scores[-4:] # or run_scores[-1: -5: -1]

# Store the most recent score in latest_score.
latest_score = run_scores[-1]

# Print the highest score among only those recent scores.
print(f"The highest score among recent scores is: {max(recent_scores)}")

# At the end, verify that pipeline_backup still contains the original five stages.
print(f"Pipeline backup : {pipeline_backup}")
print(f"Pipeline : {pipeline}")


Failed cases containing all failed records: [{'case_id': 'T002', 'category': 'safety', 'passed': False}, {'case_id': 'T003', 'category': 'hallucination', 'passed': False}, {'case_id': 'T004', 'category': 'safety', 'passed': False}, {'case_id': 'T006', 'category': 'hallucination', 'passed': False}] and total failures are: 4
Unique failed categories is : {'safety', 'hallucination'}
First failed case id is: T002
Most recent failed case is: T006
**************************************
The final Model report is: {'gpt-4': {'scores': [0.91, 0.66, 0.94], 'passed': 2, 'failed': 1}, 'claude': {'scores': [0.87, 0.72], 'passed': 1, 'failed': 1}, 'gemini': {'scores': [0.8], 'passed': 1, 'failed': 0}}
**************************************
The benchmark doesnot has complete required coverage
**************************************
Score by config: {('gpt-4', 0.2): 0.92, ('gpt-4', 0.7): 0.84, ('claude', 0.2): 0.94, ('claude', 0.7): 0.86}
Model config with score >= 0.90 : ('gpt-4', 0.2)
Model config wi

In [ ]:
# More Problem solving with DS - Lists + Tuple + Dict + Sets

# 7. [List Specific] Move invalid scores to the end
    # You receive model-evaluation scores where -1 represents a failed model call.
    # Rearrange the same list so that all -1 values move to the end, while preserving the original order of the valid scores.
scores = [0.91, -1, 0.82, -1, 0.95, 0.76, -1]

# Rules : 
    # Do not remove items from the list while looping over it.
    # Try to modify the existing scores list rather than simply hardcoding a new result.
    # There may be any number of -1s, including zero.

# Expected o/p : scores = [0.91, 0.82, 0.95, 0.76, -1, -1, -1]
# Algorithm will have two passes - 
    # [1] move all valid values to the front : [0.91, 0.82, 0.95, 0.76, 0.95, 0.76, -1]
    # [2] fill remaining to the end with -1 : [0.91, 0.82, 0.95, 0.76, -1, -1, -1]

index = 0
for score in scores:
    if(score != -1):
        scores[index]=score
        index += 1

while(index < len(scores)):
    scores[index] = -1
    index += 1

print(f"The final scores after moving invalid scores to end is: {scores}")

print("**************************************")

# 8. [Dict Specific] Model Token usage summary
# You receive daily API usage records:

usage_records = [
    {"model": "gpt-4", "tokens": 120},
    {"model": "claude", "tokens": 95},
    {"model": "gpt-4", "tokens": 150},
    {"model": "gemini", "tokens": 80},
    {"model": "claude", "tokens": 105},
    {"model": "gpt-4", "tokens": 130}
]

# Build a dict 'token_summary' with model usage info like 'total_tokens' and 'request_count'.
# Example :
    # {
    #     "gpt-4": {
    #         "total_tokens": 400,
    #         "request_count": 3
    #     }, 
    #     ...
    #     ....
    # }

token_summary = {}
for record in usage_records:
    model = record.get("model")
    tokens = record.get("tokens")

    if(model not in token_summary):
        token_summary[model] = {}
        token_summary[model]["total_token"] = 0
        token_summary[model]["request_count"] = 0

    token_summary[model]["total_token"] +=  tokens
    token_summary[model]["request_count"] += 1   

print(f"Token summary : {token_summary} ")

# After building 'token_summary', print the model with the highest total token usage.
highest_token_usage = 0
model_with_highest_usage = None
for model, details in token_summary.items():
    token_usage = details["total_token"]  # token_summary[model]["total_token"]

    if(token_usage > highest_token_usage):
        highest_token_usage = token_usage
        model_with_highest_usage = model

print(f"Model with the highest total token usage is: {model_with_highest_usage} - {highest_token_usage}")        

print("**************************************")

# 9. [Sets Specific]
# Two test suites were run against the same system:
suite_a = {"T001", "T002", "T003", "T005", "T007"}
suite_b = {"T002", "T003", "T004", "T006", "T007"}

# Using only set operations, determine:
    # test cases that were run in both suites
both_suites = suite_a & suite_b

    # test cases that were run only in suite A
only_suite_a = suite_a - suite_b

    # test cases that were run only in suite B
only_suite_b = suite_b - suite_a

    # test cases that were run in exactly one of the suites
only_one_suite_inclusive = suite_a ^ suite_b

    # all unique test cases across both suites
all_test_cases = suite_a | suite_b

# Then check if below required set of tests is fully covered by the combined test execution.
required_tests = {"T001", "T002", "T003", "T004"}

if(required_tests.issubset(all_test_cases)):
    print(f"Required set of tests is fully covered by the combined test execution")
else :
    print(f"Required set of tests is NOT fully covered by the combined test execution")    
    
# 10. [Tuple specific] Compress consecutive values

# You receive this tuple:
statuses = (
    "pass",
    "pass",
    "fail",
    "fail",
    "fail",
    "pass",
    "timeout",
    "timeout"
)

# Create a new tuple called 'compressed' containing (value, count) tuples for each consecutive group.

# Expected :
    # compressed = (
    #     ("pass", 2),
    #     ("fail", 3),
    #     ("pass", 1),
    #     ("timeout", 2)
    # )

compressed = ()
current_status = statuses[0]
count = 0

for status in statuses:
    if current_status == status:
        count += 1
    else:
        compressed = compressed + ((current_status, count),)
        current_status = status
        count = 1

# Save the final group
compressed += ((current_status, count),)

print(f"Compressed tuple is : {compressed}")
    
print("**************************************")  

# 11. [Tuple specific] Rotate a tuple to the right
# Rotate the tuple 2 positions to the right.
values = ("A", "B", "C", "D", "E", "F")

# Create 'rotated_values' and print both: values and rotated_values

# Expected : ("E", "F", "A", "B" , "C", "D")

rotated_values = ()
rotate = 2

last_2_items = values[-rotate:] # {('E', 'F')}
rest_items = values[:-rotate] # {('A', 'B', 'C', 'D')}

rotated_values = last_2_items + rest_items

print(f"Values is : {values}")
print(f"Rotated tuple by 2 positions is : {rotated_values}")       

# 12. [Tuple specific] Tuple Fundamentals Challenge
records = (
    ("gpt-4", 0.91, True),
    ("claude", 0.88, True),
    ("gemini", 0.72, False),
    ("gpt-4", 0.95, True)
)
# Each inner tuple represents: (model, score, passed)

# Part 1 — Indexing + negative indexing
    # Store: first_record , last_record using indexing.

first_record = records[0]
last_record = records[-1]

# Part 2 — Tuple unpacking
    # Unpack 'first_record' into: model score passed.
    # Then print: gpt-4 | 0.91 | True

model, score, passed = first_record
print(f"{model} | {score} | {passed}")

# Part 3 - Slicing
    # Create: 'middle_records' containing only: 
#     (
#     ("claude", 0.88, True),
#     ("gemini", 0.72, False)
# )

middle_records = records[1:-1]
print(f"Middle records: {middle_records}")

# Part 4 — Membership
    # Check whether this exact tuple exists inside records: ("gemini", 0.72, False)

if (("gemini", 0.72, False) in records):
    print("Membership exists")
else:
    print("Membership doesnot exist")


# Part 5 — Iteration + unpacking
    # Loop through 'records' and Print only the failed results.  
for (model, score, passed) in records:
    if passed is False:
        print(f"{model} failed with score {score}")

# Part 6 - .count() and .index()
labels = ("pass", "fail", "pass", "pass", "fail")
# Find - number_of_passes, first_fail_index
number_of_passes = labels.count("pass")
print(f"Number of passes: {number_of_passes}")

first_fail_index = labels.index("fail")

#Part 7 - Immutability reasoning through code
config = ("gpt-4", 0.7, "safety")

# Create a new tuple called: 'updated_config' that changes temperature from 0.7 to 0.2.
model, temperature, category = config
updated_config = (model, 0.2, category)
print(updated_config)

print("*********************************")

# 13. Distributed Job Failure Analyzer
# You receive an ordered list of execution events from a distributed job-processing system.
events = [
    {"job_id": "J001", "service": "search",   "region": "us-west", "status": "success", "latency": 120},
    {"job_id": "J002", "service": "payments", "region": "us-east", "status": "failed",  "latency": 310},
    {"job_id": "J003", "service": "search",   "region": "us-west", "status": "failed",  "latency": 250},
    {"job_id": "J002", "service": "payments", "region": "us-east", "status": "failed",  "latency": 290},
    {"job_id": "J004", "service": "search",   "region": "us-east", "status": "success", "latency": 140},
    {"job_id": "J005", "service": "payments", "region": "us-west", "status": "failed",  "latency": 450},
    {"job_id": "J006", "service": "search",   "region": "us-west", "status": "failed",  "latency": 200},
    {"job_id": "J005", "service": "payments", "region": "us-west", "status": "success", "latency": 180}
]

# A job_id may appear multiple times because jobs can be retried.

# [1] Give me Unique failed jobs by (service, region) : 
    # Build 'failures_by_group' where the key is the tuple: (service, region) and 
    # the value is a set of unique job IDs that failed at least once.

def unique_failed_jobs(events:list) -> dict:
    failures_by_group = {}

    for event in events:
        service = event.get("service")
        region = event.get("region")
            
        if (event.get("status") == "failed"):
            if (service, region) not in failures_by_group:
                failures_by_group[service, region] = set()

            failures_by_group[service, region].add(event.get("job_id"))

    return failures_by_group   

print(unique_failed_jobs(events)) # {('payments', 'us-east'): {'J002'}, ('search', 'us-west'): {'J003', 'J006'}, ('payments', 'us-west'): {'J005'}}

# [2] Create 'failure_history' containing the failed 'job_ids' in the exact chronological order in which failure events occurred.

def job_failure_history(events:list)  -> list:
    failure_history = []

    for event in events:
        if(event.get("status") == "failed"):
            failure_history.append(event.get("job_id"))

    return failure_history       

print(job_failure_history(events)) # ['J002', 'J003', 'J002', 'J005', 'J006']

# [3] Retry-recovered jobs - Create 'recovered_jobs' containing job IDs that:
# failed at least once, and later had a "success" event.

def recovered_jobs(events:list) -> set:
    recovered_jobs = set()
    job_id_failed = set()

    for event in events:
        job_id = event["job_id"]

        if(event["status"] == "failed"):
            job_id_failed.add(job_id)
        elif(event["status"] == "success" and job_id in job_id_failed):
            recovered_jobs.add(job_id)   

    return recovered_jobs       

# [4] Worst group - Determine which (service, region) combination has the largest number of unique failed jobs.

def worst_group(events:list) -> tuple:

    dict_failed_jobs = unique_failed_jobs(events)
    count = None
    worst_group_value = None

    for job,job_count in dict_failed_jobs.items():
        if(count == None):
            count = len(job_count)
            worst_group_value = job
        else:
            if(count < len(job_count)):
                count = len(job_count)
                worst_group_value = job

    return worst_group_value  # The worst group is ('search', 'us-west') with a count of 2           

# [4] Tie-breaking rule - If multiple groups have the same number of unique failed jobs, return the group whose first failure occurred earliest in events.
    # Explaination : Tie-breaking works because Python dictionaries preserve insertion order.
        # Since failures_by_group is built while processing events chronologically,
        # each (service, region) key is inserted when that group's first failure occurs.
        # worst_group() iterates the dictionary in that same order.
        # We update worst_group_value only when a group has a STRICTLY larger
        # number of unique failures (<, not <=). Therefore, if two groups tie,
        # the group whose first failure occurred earlier remains selected.

# [5] Finale question - Write function 'analyze_failures(events)' which returns all the 4 values : failures_by_group, failure_history, recovered_jobs, worst_group.
def analyze_failures(events:list) -> tuple:
    events_failures_by_group = unique_failed_jobs(events)
    events_failure_history = job_failure_history(events)
    events_recovered_jobs = recovered_jobs(events)
    events_worst_group = worst_group(events)

    return (events_failures_by_group, 
            events_failure_history, 
            events_recovered_jobs, 
            events_worst_group)


# My next exercise would be to take my above existing solution and refactor it into a single - analyze_failures(events) loop.
# What that means is instead of calling 4 separate helper functions that each loop over events, you can do one chronological pass through 
# events and build all 4 outputs at the same time. That would be the right L5 follow-up.  

def analyze_failures(events:list) -> tuple:
    failures_by_group = {}
    failure_history = []
    recovered_jobs = set()
    job_id_failed = set()

    for event in events:
        #[1] Build 'failures_by_group'
        service = event.get("service")
        region = event.get("region")
                    
        if (event.get("status") == "failed"):
            if (service, region) not in failures_by_group:
                failures_by_group[service, region] = set()
        
            failures_by_group[service, region].add(event.get("job_id"))

        # [2] Create 'failure_history'
        if(event.get("status") == "failed"):
            failure_history.append(event.get("job_id"))      

        # [3] Retry-recovered jobs
        job_id = event["job_id"]
        if(event["status"] == "failed"):
            job_id_failed.add(job_id)
        elif(event["status"] == "success" and job_id in job_id_failed):
            recovered_jobs.add(job_id)  

    # [4] Worst group 
    count = None
    worst_group_value = None
        
    for job,job_count in failures_by_group.items():
        if(count == None):
            count = len(job_count)
            worst_group_value = job
        else:
            if(count < len(job_count)):
                count = len(job_count)
                worst_group_value = job

    return (failures_by_group, 
            failure_history, 
            recovered_jobs, 
            worst_group_value)



The final scores after moving invalid scores to end is: [0.91, 0.82, 0.95, 0.76, -1, -1, -1]
**************************************
Token summary : {'gpt-4': {'total_token': 400, 'request_count': 3}, 'claude': {'total_token': 200, 'request_count': 2}, 'gemini': {'total_token': 80, 'request_count': 1}} 
Model with the highest total token usage is: gpt-4 - 400
**************************************
Required set of tests is fully covered by the combined test execution
Compressed tuple is : (('pass', 2), ('fail', 3), ('pass', 1), ('timeout', 2))
**************************************
Values is : ('A', 'B', 'C', 'D', 'E', 'F')
Rotated tuple by 2 positions is : ('E', 'F', 'A', 'B', 'C', 'D')
gpt-4 | 0.91 | True
Middle records: (('claude', 0.88, True), ('gemini', 0.72, False))
Membership exists
gemini failed with score 0.72
Number of passes: 3
('gpt-4', 0.2, 'safety')
*********************************
{('payments', 'us-east'): {'J002'}, ('search', 'us-west'): {'J003', 'J006'}, ('payments', 'u